In [1]:
import numpy as np
from pathlib import Path
from ase.build import bulk
from ase.io import write, read
from ase.calculators.espresso import Espresso, EspressoProfile
from ase.visualize import view
from ase.visualize import view as viewer
import sys

from helper import base_path, supercell_sizes, structures_path

In [4]:
# ── Build diamond ────────────────────────────────────────────

material = 'diamond'
selected_size = 1  # change as needed (0 for testing, 1 for 1x1x1, etc.)

atoms = bulk('C', 'diamond', cubic=True, a=3.567) * supercell_sizes[selected_size]
#atoms = bulk('C', 'diamond', cubic=True, a=3.567) 


save_path = structures_path / f"raw_structures/{material}_{supercell_sizes[selected_size][0]}x{supercell_sizes[selected_size][1]}x{supercell_sizes[selected_size][2]}.xyz"
save_path.parent.mkdir(parents=True, exist_ok=True)
write(save_path, atoms)

#view(atoms)

view(atoms, viewer='x3d')

In [7]:
#variables 
# 
# 
# # ── Build diamond ────────────────────────────────────────────

material = 'NV_center'
selected_size = 3  # change as needed (0 for testing, 1 for 1x1x1, etc.)

atoms = bulk('C', 'diamond', cubic=True) * supercell_sizes[selected_size]

center = atoms.get_cell().sum(axis=0) / 2
distances = np.linalg.norm(atoms.get_positions() - center, axis=1)
center_index = np.argmin(distances)

dists = atoms.get_distances(center_index, range(len(atoms)), mic=True)
dists[center_index] = 1e6
nn_index = np.argmin(dists)

del atoms[center_index]
if nn_index > center_index:
    nn_index -= 1
atoms[nn_index].symbol = 'N'

# NV center has a spin — set spin-polarized with 2 unpaired electrons
atoms.set_initial_magnetic_moments(
    [2.0 if a.symbol == 'N' else 0.0 for a in atoms]
)

save_path = structures_path / f"raw_structures/{material}_{supercell_sizes[selected_size][0]}x{supercell_sizes[selected_size][1]}x{supercell_sizes[selected_size][2]}.xyz"
save_path.parent.mkdir(parents=True, exist_ok=True)
write(save_path, atoms)


#view(atoms)
view(atoms, viewer='x3d')

In [5]:
import json
import numpy as np
from ase import Atoms
from pathlib import Path
from ase.io import write

path = "/Users/famkepotze/Desktop/3S/dft_results/relaxed_structures/optimized_tot_charge-1/NV_1x1x1.json"

with open(path) as f:
    data = json.load(f)

entry = data["1"]

# ----------------------------
# SAFE NDARRAY UNPACKER
# ----------------------------
def unpack(obj):
    return np.array(obj["__ndarray__"][2])

cell = unpack(entry["cell"]["array"]).reshape(3, 3)
positions = unpack(entry["positions"]).reshape(-1, 3)
numbers = unpack(entry["numbers"])

atoms = Atoms(
    numbers=numbers,
    positions=positions,
    cell=cell,
    pbc=True
)

save_path = Path("/Users/famkepotze/Desktop/3S/QuantumEspresso/results/relaxed_structures/NV_1x1x1.xyz")
save_path.parent.mkdir(parents=True, exist_ok=True)

write(save_path, atoms)

print("DONE →", save_path)

DONE → /Users/famkepotze/Desktop/3S/QuantumEspresso/results/relaxed_structures/NV_1x1x1.xyz


In [6]:
from ase.io import read, write
from pathlib import Path

atoms = read("/Users/famkepotze/Desktop/3S/dft_results/relaxed_structures/optimized_tot_charge-1/NV_3x3x3.json")

save_path = Path("/Users/famkepotze/Desktop/3S/dft_results/relaxed_structures/optimized_tot_charge-1/NV_3x3x3.xyz")
save_path.parent.mkdir(parents=True, exist_ok=True)

write(save_path, atoms)

In [1]:
# ============================================================
# Convert QE/ASE-style atomic structure text to XYZ coordinates
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Input file containing the structure text
# ------------------------------------------------------------
input_file = "/Users/famkepotze/Desktop/3S/dft_results/relaxed_structures/optimized_tot_charge-1/NV_3x3x3.xyz"


# ------------------------------------------------------------
# Read raw text
# ------------------------------------------------------------
with open(input_file, "r") as f:
    lines = f.readlines()

# ------------------------------------------------------------
# Parse atomic coordinates
#
# Expected format:
# Atom  x  y  z  magmom  fx  fy  fz  charge
# ------------------------------------------------------------
atoms = []

for line in lines:
    parts = line.split()

    # Skip invalid lines
    if len(parts) < 4:
        continue

    element = parts[0]

    # Only keep atomic lines
    if element not in ["C", "N", "H", "O"]:
        continue

    x = float(parts[1])
    y = float(parts[2])
    z = float(parts[3])

    atoms.append([element, x, y, z])

# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------
df = pd.DataFrame(atoms, columns=["Element", "x", "y", "z"])


save_path = Path("/Users/famkepotze/Desktop/3S/dft_results/relaxed_structures/coordinates/NV_3x3x3.csv")
save_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(save_path, sep=" ", index=False)

